# 68 — P10.7 SPIDER: internal test único y export de investigación

    **Objetivo:** evaluar una única vez el candidato congelado sobre el subset oficial de validación
    reservado como `internal_test`, sin ajustar el modelo a partir de esos resultados.

    Requiere un gate explícito:

    ```text
    PFI_P10_7_RUN_INTERNAL_TEST=YES_I_ACKNOWLEDGE_ONE_TIME
    ```


> **Gobernanza obligatoria**
>
> - No reentrena estenosis central, foraminal ni subarticular: esas tareas P10.6 ya tienen checkpoints.
> - No accede al test oculto de SPIDER.
> - No usa el `internal_test` para seleccionar modelo o ajustar hiperparámetros.
> - Antes de escribir resultados audita notebooks, manifests, resultados, modelos y todos los `.pt`.
> - Si detecta un export final/frozen previo de P10.7, aborta.
> - La salida es de investigación, requiere revisión profesional y no constituye diagnóstico clínico.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
!pip -q install timm>=1.0.0 scikit-learn>=1.3.0


In [4]:
import os

os.environ["PFI_P10_7_RUN_INTERNAL_TEST"] = (
    "YES_I_ACKNOWLEDGE_ONE_TIME"
)

print(
    "Gate:",
    os.environ.get("PFI_P10_7_RUN_INTERNAL_TEST")
)

Gate: YES_I_ACKNOWLEDGE_ONE_TIME


In [5]:
from __future__ import annotations

import hashlib
import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import timm
import torch
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, average_precision_score
from torch import nn
from torch.utils.data import DataLoader, Dataset

PFI_ROOT = Path(os.getenv("PFI_ROOT", "/content/drive/MyDrive/PFI_MVP"))
RESULTS_ROOT = Path(os.getenv(
    "PFI_P10_7_RESULTS_ROOT",
    str(PFI_ROOT / "results" / "P10_7_spider_degenerative")
))
MODELS_ROOT = Path(os.getenv(
    "PFI_P10_7_MODELS_ROOT",
    str(PFI_ROOT / "models" / "P10_7_spider_degenerative")
))

MANIFEST_PATH = RESULTS_ROOT / "disc_level_manifest_v1.csv"
BEST_CHECKPOINT = MODELS_ROOT / "checkpoints" / "best_candidate.pt"
TRAINING_SUMMARY = RESULTS_ROOT / "training_summary_v1.json"
NB67_MARKER = RESULTS_ROOT / "NOTEBOOK_67_COMPLETE.json"

EXPORT_ROOT = MODELS_ROOT / "final_internal_test_evaluation"
FROZEN_CHECKPOINT = EXPORT_ROOT / "frozen_p10_7_spider_degenerative_multitask.pt"
METRICS_PATH = EXPORT_ROOT / "internal_test_metrics_v1.json"
MODEL_CARD = EXPORT_ROOT / "MODEL_CARD.md"
SCHEMA_DRAFT = EXPORT_ROOT / "disc_degenerative_findings_schema_candidate.json"
COMPLETE_MARKER = RESULTS_ROOT / "P10_7_RESEARCH_EXPORT_COMPLETE.json"

for path in [MANIFEST_PATH, BEST_CHECKPOINT, TRAINING_SUMMARY, NB67_MARKER]:
    if not path.is_file():
        raise FileNotFoundError(f"Falta requisito: {path}")

if FROZEN_CHECKPOINT.exists() or COMPLETE_MARKER.exists():
    raise RuntimeError(
        "P10.7 ya fue evaluado/exportado. Se aborta para no repetir el internal_test."
    )

gate = os.getenv("PFI_P10_7_RUN_INTERNAL_TEST", "").strip()
if gate != "YES_I_ACKNOWLEDGE_ONE_TIME":
    raise RuntimeError(
        "Gate cerrado. Para abrir el internal_test una única vez definir "
        "PFI_P10_7_RUN_INTERNAL_TEST=YES_I_ACKNOWLEDGE_ONE_TIME"
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Gate abierto explícitamente. device:", device)


Gate abierto explícitamente. device: cuda


In [6]:
TASK_ORDER = [
    "pfirrmann_grade",
    "modic_change",
    "upper_endplate_change",
    "lower_endplate_change",
    "spondylolisthesis",
    "disc_herniation",
    "disc_narrowing",
    "disc_bulging",
]
CATEGORICAL_TASKS = {"pfirrmann_grade": 5, "modic_change": 4}
BINARY_TASKS = TASK_ORDER[2:]
task_to_index = {task: idx for idx, task in enumerate(TASK_ORDER)}

manifest = pd.read_csv(MANIFEST_PATH, dtype={"Patient": str})
internal_test = manifest[manifest["split"].eq("internal_test")].copy()
development = manifest[manifest["split"].isin(["dev_train", "dev_val"])]

if internal_test.empty:
    raise ValueError("No hay muestras internal_test.")
if set(internal_test["Patient"]) & set(development["Patient"]):
    raise ValueError("Leakage de paciente detectado.")

print("internal_test samples:", len(internal_test))
print("internal_test patients:", internal_test["Patient"].nunique())


internal_test samples: 278
internal_test patients: 39


In [7]:
class DiscCropDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        row = self.frame.iloc[index]
        payload = np.load(row["crop_path"])
        return {
            "t1": torch.from_numpy(payload["t1"]).float(),
            "t2": torch.from_numpy(payload["t2"]).float(),
            "availability": torch.from_numpy(payload["availability"]).float(),
            "ivd_index": torch.tensor(int(row["ivd_label"]) - 1, dtype=torch.long),
            "labels": torch.tensor([
                float(row[task]) if pd.notna(row[task]) else -1.0
                for task in TASK_ORDER
            ], dtype=torch.float32),
            "label_mask": torch.tensor([
                1.0 if pd.notna(row[task]) else 0.0
                for task in TASK_ORDER
            ], dtype=torch.float32),
        }

class MultiTaskDiscClassifier(nn.Module):
    def __init__(self, backbone: str, dropout: float):
        super().__init__()
        self.encoder = timm.create_model(
            backbone,
            pretrained=False,
            num_classes=0,
            global_pool="avg",
            in_chans=3,
        )
        feature_dim = int(self.encoder.num_features)
        self.ivd_embedding = nn.Embedding(25, 16)
        self.trunk = nn.Sequential(
            nn.Linear(feature_dim * 2 + 2 + 16, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.heads = nn.ModuleDict({
            "pfirrmann_grade": nn.Linear(256, 5),
            "modic_change": nn.Linear(256, 4),
            **{task: nn.Linear(256, 1) for task in BINARY_TASKS},
        })

    def forward(self, t1, t2, availability, ivd_index):
        f1 = self.encoder(t1) * availability[:, 0:1]
        f2 = self.encoder(t2) * availability[:, 1:2]
        ivd = self.ivd_embedding(ivd_index.clamp(0, 24))
        shared = self.trunk(torch.cat([f1, f2, availability, ivd], dim=1))
        return {name: head(shared) for name, head in self.heads.items()}

checkpoint = torch.load(BEST_CHECKPOINT, map_location=device, weights_only=False)
cfg = checkpoint["trainConfig"]
model = MultiTaskDiscClassifier(cfg["backbone"], cfg["dropout"]).to(device)
model.load_state_dict(checkpoint["modelStateDict"], strict=True)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)

loader = DataLoader(
    DiscCropDataset(internal_test),
    batch_size=int(cfg["batch_size"]),
    shuffle=False,
    num_workers=int(cfg["num_workers"]),
    pin_memory=torch.cuda.is_available(),
)


In [8]:
@torch.no_grad()
def evaluate_internal_test() -> dict[str, Any]:
    collected = {
        task: {"y": [], "score": [], "pred": []}
        for task in TASK_ORDER
    }

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(
            batch["t1"],
            batch["t2"],
            batch["availability"],
            batch["ivd_index"],
        )
        for task in TASK_ORDER:
            idx = task_to_index[task]
            active = batch["label_mask"][:, idx].bool()
            if not active.any():
                continue
            y = batch["labels"][active, idx].detach().cpu().numpy()
            if task in CATEGORICAL_TASKS:
                score = torch.softmax(outputs[task][active], dim=1).cpu().numpy()
                pred = score.argmax(axis=1)
            else:
                score = torch.sigmoid(outputs[task][active, 0]).cpu().numpy()
                pred = (score >= 0.5).astype(int)
            collected[task]["y"].extend(y.tolist())
            collected[task]["score"].extend(score.tolist())
            collected[task]["pred"].extend(pred.tolist())

    metrics: dict[str, Any] = {
        "schemaVersion": "pfi.p10-7-internal-test-metrics.v1",
        "status": "INTERNAL_TEST_EVALUATED_FOR_RESEARCH_EXPORT",
        "samples": int(len(internal_test)),
        "patients": int(internal_test["Patient"].nunique()),
        "tasks": {},
        "modelSelectedWithoutInternalTest": True,
        "officialHiddenTestAccessed": False,
    }

    for task, values in collected.items():
        y = np.asarray(values["y"])
        pred = np.asarray(values["pred"])
        if y.size == 0:
            metrics["tasks"][task] = {"status": "no_labels"}
            continue

        if task in CATEGORICAL_TASKS:
            metrics["tasks"][task] = {
                "macroF1": float(f1_score(y, pred, average="macro", zero_division=0)),
                "balancedAccuracy": float(balanced_accuracy_score(y, pred)),
                "support": int(y.size),
            }
        else:
            score = np.asarray(values["score"], dtype=float)
            result = {
                "f1": float(f1_score(y, pred, average="binary", zero_division=0)),
                "support": int(y.size),
            }
            if len(np.unique(y)) == 2:
                result["rocAuc"] = float(roc_auc_score(y, score))
                result["averagePrecision"] = float(average_precision_score(y, score))
            else:
                result["rocAuc"] = None
                result["averagePrecision"] = None
                result["warning"] = "single_class_in_internal_test"
            metrics["tasks"][task] = result
    return metrics

metrics = evaluate_internal_test()
print(json.dumps(metrics, indent=2, ensure_ascii=False))


{
  "schemaVersion": "pfi.p10-7-internal-test-metrics.v1",
  "status": "INTERNAL_TEST_EVALUATED_FOR_RESEARCH_EXPORT",
  "samples": 278,
  "patients": 39,
  "tasks": {
    "pfirrmann_grade": {
      "macroF1": 0.4919862763340144,
      "balancedAccuracy": 0.5209221835075494,
      "support": 278
    },
    "modic_change": {
      "macroF1": 0.31525084535228765,
      "balancedAccuracy": 0.4215223097112861,
      "support": 278
    },
    "upper_endplate_change": {
      "f1": 0.7296416938110749,
      "support": 278,
      "rocAuc": 0.7711785418392708,
      "averagePrecision": 0.7698137416140352
    },
    "lower_endplate_change": {
      "f1": 0.750788643533123,
      "support": 278,
      "rocAuc": 0.786944790369448,
      "averagePrecision": 0.793751868851867
    },
    "spondylolisthesis": {
      "f1": 0.125,
      "support": 278,
      "rocAuc": 0.7881040892193308,
      "averagePrecision": 0.08917570924148371
    },
    "disc_herniation": {
      "f1": 0.18867924528301888,
     

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [9]:
EXPORT_ROOT.mkdir(parents=True, exist_ok=False)

frozen_payload = {
    "schemaVersion": "pfi.p10-7-research-export.v1",
    "modelStateDict": checkpoint["modelStateDict"],
    "trainConfig": checkpoint["trainConfig"],
    "taskOrder": TASK_ORDER,
    "categoricalTasks": CATEGORICAL_TASKS,
    "binaryTasks": BINARY_TASKS,
    "sourceBestCandidateSha256": hashlib.sha256(BEST_CHECKPOINT.read_bytes()).hexdigest(),
    "internalTestMetrics": metrics,
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
    "autonomousDiagnosis": False,
    "officialHiddenTestAccessed": False,
}
torch.save(frozen_payload, FROZEN_CHECKPOINT)

frozen_sha = hashlib.sha256(FROZEN_CHECKPOINT.read_bytes()).hexdigest()
metrics["frozenCheckpointSha256"] = frozen_sha
METRICS_PATH.write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
    encoding="utf-8",
)

# Borrador técnico. No reemplaza el contrato pfi.degenerative-findings.v1.
schema_candidate = {
    "schemaVersion": "pfi.disc-degenerative-findings.candidate.v0",
    "status": "DRAFT_NOT_FOR_PRODUCTION",
    "findingTypes": TASK_ORDER,
    "requiredCommonFields": [
        "findingId",
        "findingType",
        "anatomy",
        "classification",
        "sourceSeries",
        "localization",
        "model",
        "review",
        "notClinicalDiagnosis",
    ],
    "reviewStatuses": ["pending", "accepted", "observed", "rejected", "edited"],
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
    "autonomousDiagnosis": False,
}
SCHEMA_DRAFT.write_text(
    json.dumps(schema_candidate, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
    encoding="utf-8",
)

MODEL_CARD.write_text(f"""# P10.7 SPIDER Degenerative Multitask — Research Export

- Frozen checkpoint SHA-256: `{frozen_sha}`
- Input: independent sagittal T1/T2 2.5D crops per IVD level
- Tasks: {", ".join(TASK_ORDER)}
- Model selection: development validation only
- Internal test: official SPIDER validation subset, opened once
- Official hidden challenge test: not accessed
- Human review required: true
- Not a clinical diagnosis: true
- Automatic anatomical localization validated: false

This export is a research artifact. Runtime integration and a versioned frontend/backend
contract remain separate work.
""", encoding="utf-8")

COMPLETE_MARKER.write_text(
    json.dumps({
        "status": "P10_7_RESEARCH_EXPORT_COMPLETE",
        "frozenCheckpointSha256": frozen_sha,
        "internalTestEvaluatedOnce": True,
        "officialHiddenTestAccessed": False,
        "readyForRuntimeIntegration": True,
        "fullProductE2EValidated": False,
    }, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print("P10_7_RESEARCH_EXPORT_COMPLETE")
print("frozen:", FROZEN_CHECKPOINT)
print("sha256:", frozen_sha)


P10_7_RESEARCH_EXPORT_COMPLETE
frozen: /content/drive/MyDrive/PFI_MVP/models/P10_7_spider_degenerative/final_internal_test_evaluation/frozen_p10_7_spider_degenerative_multitask.pt
sha256: 16eccff327e6794b127fe372ecd03ea619a0f69d939b84ae1aa2e904191c6293
